In [0]:
%sql
CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.dim_customer (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    customer_name STRING,
    city STRING,
    state STRING,
    customer_type STRING,
    credit_limit DOUBLE,
    status STRING,
    created_at TIMESTAMP,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.dim_customer (
    customer_id,
    customer_name,
    city,
    state,
    customer_type,
    credit_limit,
    status,
    created_at,
    batch_id,
    gold_loaded_at
)
SELECT
    customer_id,
    customer_name,
    city,
    state,
    customer_type,
    credit_limit,
    status,
    created_at,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM forgeaxis.forgeaxis_silver.silver_customers;


CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.dim_product (
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id STRING,
    product_name STRING,
    categories STRING,
    unit_price DOUBLE,
    manufacturing_cost DOUBLE,
    status STRING,
    created_at TIMESTAMP,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.dim_product (
    product_id,
    product_name,
    categories,
    unit_price,
    manufacturing_cost,
    status,
    created_at,
    batch_id,
    gold_loaded_at
)
SELECT
    product_id,
    product_name,
    categories,
    unit_price,
    manufacturing_cost,
    status,
    created_at,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM forgeaxis.forgeaxis_silver.silver_products;


CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.dim_machine (
    machine_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    machine_id STRING,
    machine_name STRING,
    production_line STRING,
    capacity_per_hour INT,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.dim_machine (
    machine_id,
    machine_name,
    production_line,
    capacity_per_hour,
    batch_id,
    gold_loaded_at
)
SELECT
    machine_id,
    machine_name,
    production_line,
    capacity_per_hour,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM forgeaxis.forgeaxis_silver.silver_machine;


-- date_key is the Date dimension surrogate key.
CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.dim_date (
    date_key INT,
    full_date DATE,
    day INT,
    month INT,
    quarter INT,
    year INT,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.dim_date (
    date_key,
    full_date,
    day,
    month,
    quarter,
    year,
    batch_id,
    gold_loaded_at
)
SELECT
    CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date,
    dayofmonth(full_date) AS day,
    month(full_date) AS month,
    quarter(full_date) AS quarter,
    year(full_date) AS year,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM (
    SELECT explode(
        sequence(
            DATE '2025-01-01',
            DATE '2027-12-31',
            INTERVAL 1 DAY
        )
    ) AS full_date
) AS calendar_dates;


-- ============================================================
-- FACT TABLES
-- ============================================================

CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.fact_sales_orders (
    sales_order_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    sales_order_id STRING,
    customer_sk BIGINT,
    product_sk BIGINT,
    quantity_ordered INT,
    unit_price DOUBLE,
    total_amount DOUBLE,
    order_date_key INT,
    required_date_key INT,
    order_status STRING,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.fact_sales_orders (
    sales_order_id,
    customer_sk,
    product_sk,
    quantity_ordered,
    unit_price,
    total_amount,
    order_date_key,
    required_date_key,
    order_status,
    batch_id,
    gold_loaded_at
)
SELECT
    s.sales_order_id,
    c.customer_sk,
    p.product_sk,
    s.quantity_ordered,
    s.unit_price,
    s.total_amount,
    CAST(date_format(s.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
    CAST(date_format(s.required_date, 'yyyyMMdd') AS INT) AS required_date_key,
    s.order_status,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM forgeaxis.forgeaxis_silver.silver_sales_orders AS s
INNER JOIN forgeaxis.forgeaxis_gold.dim_customer AS c
    ON s.customer_id = c.customer_id
INNER JOIN forgeaxis.forgeaxis_gold.dim_product AS p
    ON s.product_id = p.product_id;


CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.fact_production_orders (
    production_order_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    production_order_id STRING,
    sales_order_id STRING,
    machine_sk BIGINT,
    quantity_produced INT,
    start_date_key INT,
    end_date_key INT,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    production_status STRING,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.fact_production_orders (
    production_order_id,
    sales_order_id,
    machine_sk,
    quantity_produced,
    start_date_key,
    end_date_key,
    start_time,
    end_time,
    production_status,
    batch_id,
    gold_loaded_at
)
SELECT
    p.production_order_id,
    p.sales_order_id,
    m.machine_sk,
    p.quantity_produced,
    CAST(date_format(p.start_time, 'yyyyMMdd') AS INT) AS start_date_key,
    CAST(date_format(p.end_time, 'yyyyMMdd') AS INT) AS end_date_key,
    p.start_time,
    p.end_time,
    p.production_status,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM forgeaxis.forgeaxis_silver.silver_production_orders AS p
INNER JOIN forgeaxis.forgeaxis_gold.dim_machine AS m
    ON p.machine_id = m.machine_id;


CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.fact_quality_checks (
    quality_check_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    quality_check_id STRING,
    production_order_id STRING,
    inspector_name STRING,
    inspection_date_key INT,
    inspection_time TIMESTAMP,
    defect_type STRING,
    defect_count INT,
    quality_status STRING,
    remarks STRING,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.fact_quality_checks (
    quality_check_id,
    production_order_id,
    inspector_name,
    inspection_date_key,
    inspection_time,
    defect_type,
    defect_count,
    quality_status,
    remarks,
    batch_id,
    gold_loaded_at
)
SELECT
    quality_check_id,
    production_order_id,
    inspector_name,
    CAST(date_format(inspection_time, 'yyyyMMdd') AS INT) AS inspection_date_key,
    inspection_time,
    defect_type,
    defect_count,
    quality_status,
    remarks,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM forgeaxis.forgeaxis_silver.silver_quality_checks;


CREATE OR REPLACE TABLE forgeaxis.forgeaxis_gold.fact_sensor_logs (
    sensor_log_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    sensor_log_id STRING,
    production_order_id STRING,
    machine_sk BIGINT,
    date_key INT,
    sensor_timestamp TIMESTAMP,
    temperature DOUBLE,
    pressure DOUBLE,
    vibration DOUBLE,
    humidity DOUBLE,
    batch_id STRING,
    gold_loaded_at TIMESTAMP
)
USING DELTA;

INSERT INTO forgeaxis.forgeaxis_gold.fact_sensor_logs (
    sensor_log_id,
    production_order_id,
    machine_sk,
    date_key,
    sensor_timestamp,
    temperature,
    pressure,
    vibration,
    humidity,
    batch_id,
    gold_loaded_at
)
SELECT
    s.sensor_log_id,
    s.production_order_id,
    m.machine_sk,
    CAST(date_format(s.`timestamp`, 'yyyyMMdd') AS INT) AS date_key,
    s.`timestamp` AS sensor_timestamp,
    s.temperature,
    s.pressure,
    s.vibration,
    s.humidity,
    concat(
        'gold_full_load_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS batch_id,
    current_timestamp() AS gold_loaded_at
FROM forgeaxis.forgeaxis_silver.silver_sensor_logs AS s
INNER JOIN forgeaxis.forgeaxis_gold.dim_machine AS m
    ON s.machine_id = m.machine_id;

In [0]:
%sql
SELECT customer_sk, customer_id, customer_name
FROM forgeaxis.forgeaxis_gold.dim_customer
LIMIT 10;

Raw CSV
→ Bronze
→ Silver
→ Gold full load
→ Incremental pipeline
→ Watermark + MERGE
→ Workflow + retry handling
→ End-to-end scheduled pipeline
→ SQL views
→ Dashboard

In [0]:
%sql
select * from forgeaxis.forgeaxis_gold.fact_production_orders;